# Blood Pressure Estimation with Neural Networks

In this notebook it is possible to find a tutorial on how to employ the trained neural-network-based model.
The netobook resume the trained model and simply run inference on a sample from the test set. The purpose is only to clarify how the model should be utilized inside an application.

![Model Inference Visualization](inference_schema.jpg)

## Setup Environment

In [2]:
import os
import sys
import yaml
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
folders_to_add = ['data', 'models', 'training_utils']
for folder in folders_to_add:
    folder_path = os.path.join(project_root, folder)
    if folder_path not in sys.path:
        sys.path.insert(0, folder_path)
import torch
import numpy as np
from data.dataset import PhysioDataset
from data.preprocessing_utils.data_visualization import plot_signals
from models import ResGRUNet

ModuleNotFoundError: No module named 'data'

In [ ]:
# Load the YAML configuration file to get the trianing configuration
# N.B. replace path with the actual path to your checkpoint/YAML file

checkpoint_path = '../checkpoints/vanilla_resnet/vanilla_resnet-ResGRUNet-2025_05_24-15_03_03/vanilla_resnet/ckpt/ResGRUNet'
config_file_path = '../checkpoints/vanilla_resnet/vanilla_resnet-ResGRUNet-2025_05_24-15_03_03/vanilla_resnet/ckpt/config.yaml'  

with open(config_file_path, 'r') as file:
    config = yaml.safe_load(file)

# Print the loaded configuration to verify
print(config)

 # Select the gpu to be usesd
os.environ['CUDA_VISIBLE_DEVICES'] = "0"
device = torch._C.device("cuda:0")

## Load Dataset

dataset = PhysioDataset(
        seed=config['seed'],
        lmdb_folder=os.path.join(config['dataset_folder'], config['dataset_name']),
        pretraining_ratio=config['pretraining_ratio'],
        pretraining_split_ratio=list(map(float, config['pretraining_tr_val_tt_split_ratio'].split(','))),
        personalization_sample_number=config['personalization_sample_number'],
        mix_pretraining_subject_samples=config['mix_pretraining_subject_samples'],
        fs=config['fs'],
        input_seq_len_s=config['input_seq_len_s'],
        ecg=config['ecg'],
        resp=config['resp'],
        ppg_derivatives=config['ppg_derivatives'],
        ppg_emd=config['ppg_emd'],
        ppg_freqs=config['ppg_freqs']
    )

 # Get train/val/test samplers and build the dataloaders
(train_sampler, val_sampler, test_sampler) = dataset.get_pretraining_samplers()

test_dataloader = DataLoader(dataset, sampler=test_sampler, batch_size=config['batchsize'], num_workers=config['loader_worker'], pin_memory=True)

## Load Model

In [ ]:
model = ResGRUNet.ResGRUNet(
            ecg=config['ecg'],
            resp=config['resp'], 
            ppg_derivatives=config['ppg_derivatives'], 
            ppg_emd=config['ppg_emd'], 
            ppg_freqs=config['ppg_freqs'],
            channels=config['channels'], 
            kernel_size=config['kernel_size'], 
            act=config['act'], 
            pooling=config['pooling'], 
            proj_head_dim=config['proj_head_dim'], 
            input_seq_len=int(config['input_seq_len_s'] * config['fs']),
            return_embedding=config['return_embedding']
        )

checkpoint = torch.load(checkpoint_path, weights_only=False)
model.load_state_dict(checkpoint['model'])
model.eval()
model = model.to(device)

# Test Model

In [ ]:
# Get a single batch from the test dataloader
batch = next(iter(test_dataloader))

# Extract signals and targets from the batch
signals, targets = batch

# Move data and labels to the same device as the model
signals = signals.to(device)
targets = torch.cat((batch[1][0].to(device), batch[1][1].to(device)), dim=-1)

# Perform forward pass
if config['return_embedding']:
    outputs, _ = model(signals)
else:
    outputs = model(signals)

# Print the outputs for verification
print("Model Outputs:", outputs)

Visualize prediction results on a sample from the batch.

In [ ]:
idx = 3  # Index of the sample to visualize

signals = signals.cpu().numpy()

targets = targets.cpu().numpy()
sbp_val = targets[idx, 0].item()
dbp_val = targets[idx, 1].item()

outputs = outputs.cpu().detach().numpy()
out_sbp_val = outputs[idx, 0].item()
out_dbp_val = outputs[idx, 1].item()

plot_signals(
            signals[idx, :, :].T, fs=config['fs'], 
            labels=['PPG', 'ECG'], 
            title=f'Ground Truth [SBP {sbp_val:.2f} - DBP {dbp_val:.2f}], Predicted [SBP {out_sbp_val:.2f} - DBP {out_dbp_val:.2f}]', 
            savepath='./prediction_example.jpg', 
            ylabels=['a.u.', 'mV']
            )